In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import xgboost as xgb
from sklearn.base import clone
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/playground-series-s6e2/sample_submission.csv
/kaggle/input/playground-series-s6e2/train.csv
/kaggle/input/playground-series-s6e2/test.csv
/kaggle/input/heartdisease/Heart_Disease_Prediction.csv


In [2]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

CONFIG = config()

In [3]:
train = pd.read_csv(f'{CONFIG.INPUT_DIR}/train.csv')
train['source'] = 'train'
test = pd.read_csv(f'{CONFIG.INPUT_DIR}/test.csv')
test['source'] = 'test'
sample_sub = pd.read_csv(f'{CONFIG.INPUT_DIR}/sample_submission.csv')

org = pd.read_csv('/kaggle/input/heartdisease/Heart_Disease_Prediction.csv')
org['source'] = 'original'

In [4]:
for df, name in zip([train, test, org], ['train', 'test', 'original']):
    print(f'NULL VALUE COUNTS FOR {name}:')
    print(df.isnull().sum())
    print('='*30)
    print(f'{name} shape:')
    print(df.shape)
    print('='*30)
    if name == 'train':
        print('General EDA -- TRAIN ONLY', end='\n')
        print(f'Dtypes :', end='\n')
        print(df.dtypes)
        print('='*30)
        print(f'NUMBER OF UNIQUE VALUES :', end='\n')
        print(df.nunique())
        print('='*30)
    

NULL VALUE COUNTS FOR train:
id                         0
Age                        0
Sex                        0
Chest pain type            0
BP                         0
Cholesterol                0
FBS over 120               0
EKG results                0
Max HR                     0
Exercise angina            0
ST depression              0
Slope of ST                0
Number of vessels fluro    0
Thallium                   0
Heart Disease              0
source                     0
dtype: int64
train shape:
(630000, 16)
General EDA -- TRAIN ONLY
Dtypes :
id                           int64
Age                          int64
Sex                          int64
Chest pain type              int64
BP                           int64
Cholesterol                  int64
FBS over 120                 int64
EKG results                  int64
Max HR                       int64
Exercise angina              int64
ST depression              float64
Slope of ST                  int64
Number of ves

In [5]:
combine = pd.concat([train.drop(columns='id'), test.drop(columns='id'), org], ignore_index=True).reset_index()
combine.shape

(900270, 16)

In [6]:
train['Heart Disease'].unique()

array(['Presence', 'Absence'], dtype=object)

In [7]:
class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

FEATURES = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source']]
X = train[FEATURES]
X_org = org[FEATURES]

y = train[CONFIG.TARGET].map(class_mapping)
y_org = org[CONFIG.TARGET].map(class_mapping)

X_test = test[FEATURES]

In [8]:
xgb_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'eval_metric': 'auc',
    'early_stopping_rounds': 100,
    'random_state': CONFIG.SEED
}

In [9]:
skf = StratifiedKFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)
kf = KFold(n_splits=CONFIG.N_FOLDS, random_state=CONFIG.SEED, shuffle=True)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for i, (train_idx, val_idx) in enumerate(skf.split(X, y), 0):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    X_org_n = X_org.copy()
    y_org_n = y_org.copy()
    X_org_n = pd.concat([X_org_n]*10, axis=0, ignore_index=True)
    y_org_n = pd.concat([y_org_n]*10, axis=0, ignore_index=True)
    X_train = pd.concat([X_train, X_org_n], axis=0, ignore_index=True)
    y_train = pd.concat([y_train, y_org_n], axis=0, ignore_index=True)
    model = clone(xgb.XGBClassifier(**xgb_params))
    model.fit(X_train, y_train,
             eval_set=[(X_val, y_val)], 
             verbose=200)
    preds = model.predict_proba(X_val)[:,1]
    oof_preds[val_idx] = preds
    test_preds += (model.predict_proba(X_test)[:, 1] / CONFIG.N_FOLDS)
    print(f'SCORE FOR FOLD{i} : {roc_auc_score(y_val, preds)}')
    
overall_roc_auc = roc_auc_score(y, oof_preds)
print(f'SCORE ACROSS ALL FOLDS : {overall_roc_auc}')

[0]	validation_0-auc:0.94009
[200]	validation_0-auc:0.95496
[400]	validation_0-auc:0.95543
[600]	validation_0-auc:0.95545
[630]	validation_0-auc:0.95544
SCORE FOR FOLD0 : 0.9554608574043791
[0]	validation_0-auc:0.93820
[200]	validation_0-auc:0.95410
[400]	validation_0-auc:0.95451
[531]	validation_0-auc:0.95450
SCORE FOR FOLD1 : 0.9545180745987629
[0]	validation_0-auc:0.93861
[200]	validation_0-auc:0.95482
[400]	validation_0-auc:0.95528
[600]	validation_0-auc:0.95529
[686]	validation_0-auc:0.95525
SCORE FOR FOLD2 : 0.9552973090123478
[0]	validation_0-auc:0.93878
[200]	validation_0-auc:0.95439
[400]	validation_0-auc:0.95484
[583]	validation_0-auc:0.95482
SCORE FOR FOLD3 : 0.9548532152974598
[0]	validation_0-auc:0.93965
[200]	validation_0-auc:0.95509
[400]	validation_0-auc:0.95561
[600]	validation_0-auc:0.95561
[623]	validation_0-auc:0.95560
SCORE FOR FOLD4 : 0.9556427003540627
SCORE ACROSS ALL FOLDS : 0.9551528112052853


In [10]:
sample_sub['Heart Disease'] = test_preds
sample_sub.to_csv(f'submission_{overall_roc_auc}.csv', index=False)